# Day 5: Session 5B - Several Answers at Once

[Session Webpage](https://eds-217-essential-python.github.io/course-materials/interactive-sessions/5b_aggregating_data.html)

Date: 09/04/2026

In [1]:
import pandas as pd

url = 'https://eds-217-essential-python.github.io/data/national_parks.csv'
parks = pd.read_csv(url)

# Remove the rows that have park Totals in them and make a new dataframe:
by_year = parks[~(parks['year'] == 'Total')]
national_parks = by_year[by_year['unit_type'] == 'National Park'].copy()

national_parks.shape

(4682, 12)

In [2]:
national_parks.groupby('region')['visitors'].mean()

region
AK    1.204881e+05
IM    7.931557e+05
MW    6.108661e+05
NC    5.107866e+05
NE    1.618013e+06
PW    6.627740e+05
SE    1.572087e+06
Name: visitors, dtype: float64

In [3]:
national_parks.groupby('region')['visitors'].count()

region
AK     488
IM    1590
MW     533
NC      46
NE     179
PW    1393
SE     453
Name: visitors, dtype: int64

### pass a list to .agg() to get several summaries of one column


In [4]:
# .agg() takes a list of aggregating commands to run all at once
# 'count', 'mean', 'sum', 'min', 'max', 'median', 'std', 'nunique'.

national_parks.groupby('region')['visitors'].agg(['count', 'mean', 'max'])
#returns a dataframe

,count,mean,max
region,,,
AK,488,1.204881e+05,592431.0
IM,1590,7.931557e+05,5969811.0
MW,533,6.108661e+05,3527837.0
NC,46,5.107866e+05,680862.0
NE,179,1.618013e+06,5440952.0
PW,1393,6.627740e+05,5028868.0
SE,453,1.572087e+06,11312786.0


In [5]:
national_parks.groupby('region')['unit_name'].nunique()

region
AK     9
IM    18
MW     7
NC     1
NE     2
PW    17
SE     7
Name: unit_name, dtype: int64

In [6]:
# Practice
national_parks.groupby('state')['visitors'].agg(['count', 'mean', 'max'])

,count,mean,max
state,,,
AK,488,1.204881e+05,592431.0
AR,109,8.443021e+05,2092400.0
AS,13,8.745692e+03,28892.0
AZ,290,1.009639e+06,5969811.0
CA,789,6.076323e+05,5028868.0
CO,378,6.353273e+05,4517585.0
FL,194,3.961249e+05,1534328.0
HI,153,8.838688e+05,2247974.0
KY,81,1.046299e+06,2396234.0


### pass a dictionary to .agg() to summarise several columns differently


In [7]:
# passing a list through .agg() runs the same agg on one column
# Often you want something else: the mean of this column, 
# the sum of that one, the number of distinct values in a third...

# .agg() takes a dictionary. 
# The keys are column names and the values are the agg methods
# {'col_name' : 'agg method'}

national_parks.groupby('region').agg({
    'visitors': 'mean',
    'unit_name': 'nunique',
    'year': 'min'
})
# the result has one column per key, named after the key

,visitors,unit_name,year
region,,,
AK,1.204881e+05,9,1922
IM,7.931557e+05,18,1904
MW,6.108661e+05,7,1904
NC,5.107866e+05,1,1971
NE,1.618013e+06,2,1919
PW,6.627740e+05,17,1904
SE,1.572087e+06,7,1931


In [8]:
# Practice: Write one .agg() call, grouped by region, 
# that reports the median number of visitors, 
# the latest year of record, and 
# the number of distinct states in each region

national_parks.groupby('region').agg({
    'visitors': 'median',
    'year': 'max',
    'state': 'nunique'
})

,visitors,year,state
region,,,
AK,24595.0,2016,1
IM,378050.0,2016,7
MW,404700.0,2016,6
NC,538297.0,2016,1
NE,1699228.0,2016,2
PW,407653.0,2016,6
SE,513397.0,2016,5


### apply the top-N pattern to a grouped result


In [9]:
# df.sort_values('column', ascending=False).head(n)
national_parks.groupby(
    'unit_name')['visitors'].mean().sort_values(
        ascending=True).head(10)


unit_name
Kobuk Valley National Park            4475.857143
Gates of the Arctic National Park     6451.800000
National Park of American Samoa       8745.692308
Lake Clark National Park             10396.971429
Isle Royale National Park            13917.064935
Dry Tortugas National Park           25057.075000
Katmai National Park                 25486.225806
Great Basin National Park            45431.012048
Wrangell-St. Elias National Park     45923.771429
Congaree National Park               82177.812500
Name: visitors, dtype: float64

In [10]:
## When the grouped result is a DataFrame rather than a Series, 
# .sort_values() needs to be told which column to sort on, 
# exactly as it does on a raw table:

park_stats = national_parks.groupby('unit_name')['visitors'].agg(
    ['count', 'mean', 'max'])

park_stats.sort_values('mean', ascending=False).head(10)
park_stats.loc['Grand Canyon National Park']

count    9.800000e+01
mean     2.096805e+06
max      5.969811e+06
Name: Grand Canyon National Park, dtype: float64


### explain why a grouped mean should always be reported with the count it was computed from

In [11]:
# Practice: 

# Filter national_parks to the single year '2016', 
filtered = national_parks[national_parks['year'] == '2016']
# then group by state
grouped = filtered.groupby('state')
# use .agg() to report the count and mean of visitors. 
aggregated = grouped['visitors'].agg(['count', 'mean'])
# Rank the result by mean.
ranked = aggregated.sort_values('mean')
ranked

,count,mean
state,,
MI,1,2.496600e+04
AS,1,2.889200e+04
SC,1,1.438430e+05
NV,1,1.448460e+05
MN,1,2.419120e+05
AK,9,2.450048e+05
TX,2,2.850645e+05
VI,1,4.113430e+05
NM,1,4.667730e+05


In [12]:
## Same same but different 
(national_parks[
    national_parks['year'] == '2016']
    .groupby('state')['visitors']
    .agg(['count', 'mean'])
    .sort_values('mean')
    )

,count,mean
state,,
MI,1,2.496600e+04
AS,1,2.889200e+04
SC,1,1.438430e+05
NV,1,1.448460e+05
MN,1,2.419120e+05
AK,9,2.450048e+05
TX,2,2.850645e+05
VI,1,4.113430e+05
NM,1,4.667730e+05


### recognize a MultiIndex produced by grouping on two keys, and flatten it with .reset_index()

In [13]:
## Grouping by two keys
two_keys = national_parks.groupby(['region', 'state'])['visitors'].mean()
two_keys

region  state
AK      AK       1.204881e+05
IM      AZ       1.009639e+06
        CO       6.353273e+05
        MT       9.908449e+05
        NM       4.702549e+05
        TX       1.935455e+05
        UT       5.792425e+05
        WY       1.606659e+06
MW      AR       8.443021e+05
        MI       1.391706e+04
        MN       2.161420e+05
        ND       4.142048e+05
        OH       2.093105e+06
        SD       5.786315e+05
NC      VA       5.107866e+05
NE      ME       1.668492e+06
        VA       1.556940e+06
PW      AS       8.745692e+03
        CA       6.076323e+05
        HI       8.838688e+05
        NV       4.543101e+04
        OR       3.068096e+05
        WA       1.115853e+06
SE      FL       3.961249e+05
        KY       1.046299e+06
        NC       6.069152e+06
        SC       8.217781e+04
        VI       4.330022e+05
Name: visitors, dtype: float64

In [14]:
print(two_keys.index)
#A MultiIndex is genuinely useful, 
# and it is also a common reason that pandas code stops working, 
# because everything you know about selecting from an index now needs a tuple

MultiIndex([('AK', 'AK'),
            ('IM', 'AZ'),
            ('IM', 'CO'),
            ('IM', 'MT'),
            ('IM', 'NM'),
            ('IM', 'TX'),
            ('IM', 'UT'),
            ('IM', 'WY'),
            ('MW', 'AR'),
            ('MW', 'MI'),
            ('MW', 'MN'),
            ('MW', 'ND'),
            ('MW', 'OH'),
            ('MW', 'SD'),
            ('NC', 'VA'),
            ('NE', 'ME'),
            ('NE', 'VA'),
            ('PW', 'AS'),
            ('PW', 'CA'),
            ('PW', 'HI'),
            ('PW', 'NV'),
            ('PW', 'OR'),
            ('PW', 'WA'),
            ('SE', 'FL'),
            ('SE', 'KY'),
            ('SE', 'NC'),
            ('SE', 'SC'),
            ('SE', 'VI')],
           names=['region', 'state'])


In [15]:
# Need to know how to recognize one and get out of it, 
# and the way out is a single method:
flat = two_keys.reset_index()
flat.head()
# .reset_index() takes the index levels and turns them back into ordinary columns. 
# The result is a plain DataFrame with three columns, 
# and everything you have learned this week works on it again:

,region,state,visitors
0,AK,AK,1.204881e+05
1,IM,AZ,1.009639e+06
2,IM,CO,6.353273e+05
3,IM,MT,9.908449e+05
4,IM,NM,4.702549e+05


#### When to reach for .reset_index()
##### Whenever a grouped result is going to be used rather than read. If you are filtering it, merging it, plotting it, or writing it to a file, flatten it first. If you are looking at it in a notebook cell and moving on, leave it alone.

##### .reset_index() also works on a single-key grouped result, and turns the Series into a two-column DataFrame, which is often exactly what you want before plotting.

## Putting it all together

In [16]:
pacific_west = national_parks[national_parks['region'] == 'PW']

pw_stats = pacific_west.groupby('unit_name')['visitors'].agg(['count', 'mean', 'max'])

pw_stats.sort_values('mean', ascending=False).head(10)

,count,mean,max
unit_name,,,
Olympic National Park,82,1.960219e+06,3846709.0
Yosemite National Park,111,1.715356e+06,5028868.0
Haleakala National Park,57,9.164551e+05,1963187.0
Hawai'i Volcanoes National Park,96,8.645207e+05,2247974.0
Mount Rainier National Park,113,8.361841e+05,1925100.0
Joshua Tree National Park,76,7.458512e+05,2505286.0
Sequoia National Park,111,5.662813e+05,1254688.0
Death Valley National Park,84,5.477119e+05,1296283.0
Kings Canyon National Park,113,4.688335e+05,1216800.0


### Key points
.agg(['count', 'mean', 'max']) runs several summaries on one column and returns a DataFrame.

.agg({'col': 'fn', 'other': 'fn'}) runs different summaries on different columns. There is no ['column'] before it, because the dictionary keys name the columns.

Put 'count' first, every time. A grouped mean without a count beside it is half a number.

The top-N pattern works on a grouped result: .groupby(...)...sort_values(...).head(n).

On a grouped DataFrame, .sort_values() needs the name of the column to sort by.

Grouping by a list of keys produces a MultiIndex.

.reset_index() turns index levels back into columns. Use it whenever the result is going to be used rather than just read.